# Лабораторная работа № 6

## Обратная задача кинематики

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 7. Обратная задача кинематики как задача решения нелинейной системы

---

**Студент:** _фамилия, имя, группа_
**Дата:**

---

### Цель работы

Решить обратную задачу кинематики двумя способами — аналитически и методом Ньютона, —
исследовать бассейны притяжения численного метода, увидеть, что происходит вблизи
границы рабочей зоны, и разобраться с избыточным манипулятором.

### Что понадобится

`numpy`, `scipy`, `matplotlib`.

### Правило практикума

Метод Ньютона вы реализуете сами. `scipy.optimize.least_squares` используется
в задании 6 по прямому назначению — как промышленный решатель с ограничениями,
и результат сравнивается с собственной реализацией.

## Теоретическая справка

### Постановка

Прямая задача кинематики — отображение $f: T^n \to SE(3)$, вычисляемое произведением
матриц вдоль цепи. Обратная задача — найти конфигурацию по заданной позе схвата:

$$\text{найти } q \text{ такое, что } f(q) = x^* .$$

Симметрии между задачами нет никакой, и это главное, что нужно вынести из работы.

| | Прямая задача | Обратная задача |
|---|---|---|
| Решение существует | всегда | не всегда |
| Решение единственно | всегда | почти никогда |
| Явная формула | есть | только для частных структур |
| Сложность | линейная по числу звеньев | нелинейная система |

### Плоский 2R: аналитическое решение

Для двухзвенника решение выписывается через теорему косинусов. Если
$r^2 = x^2 + y^2$, то

$$\cos q_2 = \frac{r^2 - l_1^2 - l_2^2}{2 l_1 l_2},
\qquad q_1 = \operatorname{atan2}(y, x) - \operatorname{atan2}\bigl(l_2 \sin q_2,\; l_1 + l_2\cos q_2\bigr).$$

Знак $q_2$ выбирается произвольно — отсюда **две ветви решения**, которые называют
«локоть вверх» и «локоть вниз». Условие достижимости:
$|l_1 - l_2| \le r \le l_1 + l_2$, то есть рабочая зона — кольцо.

### Численное решение: метод Ньютона

В общем случае замкнутой формулы нет, и задачу решают как систему нелинейных
уравнений $F(q) = f(q) - x^* = 0$. Шаг метода Ньютона:

$$J(q)\,\Delta q = -F(q), \qquad q \leftarrow q + \Delta q,$$

где $J = \partial f / \partial q$ — якобиан. Для плоского 2R он выписывается явно,
и его определитель равен

$$\det J = l_1 l_2 \sin q_2 .$$

Определитель обращается в нуль при $q_2 = 0$ и $q_2 = \pi$ — рука полностью вытянута
или сложена. Это **сингулярные конфигурации**: именно там метод начинает буксовать,
и подробно ими займёмся в теме 8.

### Демпфирование

Вблизи вырождения шаг Ньютона неограниченно растёт. Лекарство — регуляризация
Левенберга: вместо $J\Delta q = -F$ решается

$$(J^{\mathsf T}J + \lambda^2 I)\,\Delta q = -J^{\mathsf T}F .$$

Шаг остаётся ограниченным, но решение смещается: при $\lambda > 0$ невязка
в точности нулю уже не равна. Это типичный размен устойчивости на точность.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

L1, L2 = 0.5, 0.35

## Задание 1. Прямая кинематика и якобиан

Реализуйте `fk2(q)` для плоского 2R с $l_1 = 0{,}5$, $l_2 = 0{,}35$ и якобиан
`jac2(q)` аналитически. Проверьте его центральными разностями.

**Контроль.** Расхождение аналитического и численного якобиана на уровне $10^{-11}$.
Определитель должен совпасть с $l_1 l_2 \sin q_2$ с машинной точностью.

**Вопрос для вывода:** при каких $q_2$ определитель обращается в нуль и что это
означает геометрически?

In [ ]:
# TODO: задание 1

## Задание 2. Аналитическое решение: две ветви

Реализуйте `ik2_analytic(target)`, возвращающую **список** решений: пустой,
если точка недостижима, и два решения иначе.

Проверьте обе ветви подстановкой в прямую кинематику. Определите границы рабочей зоны
и убедитесь, что для точки вне кольца решений нет.

**Контроль.** Для цели $(0{,}55;\ 0{,}30)$ два решения с нулевой невязкой; для
$(0{,}95;\ 0)$ решений нет; кольцо от 0,15 до 0,85 м.

**Вопросы для вывода:** почему решений ровно два, а не одно и не три? Сколько решений
у точки, лежащей точно на границе кольца?

In [ ]:
# TODO: задание 2

## Задание 3. Метод Ньютона и скорость сходимости

Реализуйте `ik2_newton(target, q0, damp)` — собственный метод Ньютона
с необязательным демпфированием. Верните не только решение, но и историю невязок.

Запустите из точки $q_0 = (0;\ 1{,}2)$ и выпишите невязку по итерациям. Оцените
отношение $r_{k+1}/r_k^2$ в той фазе, где невязка уже мала.

**Контроль.** Сходимость за 5 итераций до машинной точности; отношение
$r_{k+1}/r_k^2$ порядка единицы.

**Вопросы для вывода:** какова скорость сходимости и как это видно из последовательности
невязок? К какой из двух ветвей сошёлся метод и почему именно к ней?

In [ ]:
# TODO: задание 3

## Задание 4. Бассейны притяжения

Возьмите сетку $220 \times 220$ начальных приближений на всём конфигурационном
пространстве $[-\pi, \pi]^2$. Для каждого запустите метод Ньютона и определите,
к какой ветви он сошёлся (или не сошёлся вовсе).

Постройте две картинки: карту ветвей и карту числа итераций.

**Контроль.** Обе ветви должны собрать примерно по половине площади; доля
несошедшихся мала.

**Вопросы для вывода:** выглядит ли граница между бассейнами гладкой? Что происходит
вблизи прямой $q_2 = 0$ и почему? Можно ли по начальному приближению заранее
предсказать ветвь?

In [ ]:
# TODO: задание 4

## Задание 5. Что происходит вблизи границы рабочей зоны

Возьмите цели на расстоянии $r = (l_1 + l_2) - \delta$ от начала координат
для $\delta$ от $10^{-1}$ до $10^{-6}$. Для каждой посчитайте число обусловленности
якобиана в точке решения и число итераций метода.

Оцените наклон зависимости $\log(\operatorname{cond} J)$ от $\log \delta$.

Затем возьмите цель **точно** на границе и сравните обычный метод Ньютона
с демпфированным при $\lambda = 10^{-2}$.

**Контроль.** Наклон близок к $-0{,}5$. На границе демпфированный метод не достигает
машинной точности.

**Вопросы для вывода:** почему обусловленность растёт именно как $\delta^{-1/2}$?
Что демпфирование даёт и чем за это приходится платить?

In [ ]:
# TODO: задание 5

## Задание 6. Промышленный решатель с ограничениями

Решите ту же задачу через `scipy.optimize.least_squares`, передав аналитический
якобиан и ограничения $q_2 \in [0{,}05;\ \pi]$ — то есть запретив ветвь «локоть вниз».

Затем задайте заведомо недостижимую цель $(1{,}2;\ 0)$ и посмотрите, что вернёт
решатель.

**Контроль.** С ограничениями находится ветвь «локоть вверх». Для недостижимой цели
невязка равна расстоянию от цели до границы рабочей зоны.

**Вопросы для вывода:** как ограничения на суставные углы помогают выбрать нужную
ветвь? Что геометрически означает решение для недостижимой цели?

In [ ]:
# TODO: задание 6

## Задание 7. Избыточный манипулятор: ядро якобиана

Для плоского 3R с длинами $(0{,}5;\ 0{,}35;\ 0{,}2)$ задача о выводе схвата
в точку имеет два уравнения и три неизвестных — решений бесконечно много.

Решите её методом Ньютона с псевдообращением якобиана. Затем найдите вектор,
порождающий ядро якобиана (правый сингулярный вектор при нулевом сингулярном числе),
и проверьте, что движение вдоль него почти не смещает схват.

**Контроль.** Невязка на уровне $10^{-16}$; размерность ядра равна 1; сдвиг на 0,05 рад
вдоль ядра смещает схват меньше чем на 0,2 мм.

**Вопросы для вывода:** почему при сдвиге на 0,2 рад смещение уже около 1,5 мм — разве
движение вдоль ядра не должно сохранять положение схвата точно? Как этим
пространством решений распорядиться с пользой?

In [ ]:
# TODO: задание 7

## Выводы

_Ответьте на все вопросы, отмеченные как «вопрос для вывода», приводя конкретные числа._

## Контрольные вопросы

1. Сопоставьте прямую и обратную задачи кинематики по четырём признакам:
   существование, единственность, наличие формулы, сложность.
2. Выведите аналитическое решение для плоского 2R через теорему косинусов.
3. Почему решений ровно два? Какие структуры манипуляторов допускают решение
   в замкнутой форме?
4. Запишите шаг метода Ньютона для обратной кинематики. Что такое якобиан
   и какова его размерность для плоского 2R?
5. Чему равен $\det J$ для плоского 2R? При каких конфигурациях он обращается в нуль?
6. Что такое бассейн притяжения? Почему граница между бассейнами не гладкая?
7. Как ведёт себя число обусловленности якобиана при приближении к границе
   рабочей зоны и почему показатель равен $-1/2$?
8. В чём состоит регуляризация Левенберга и какой ценой она даёт устойчивость?
9. Что такое избыточный манипулятор? Какова размерность множества решений
   для плоского 3R при задании только положения схвата?
10. Почему движение вдоль ядра якобиана сохраняет положение схвата лишь приближённо?